Crude Integration of Structural Featurizer

In [2]:
import sys 
sys.path.append('/Users/amalyajohnson/Desktop/SpectraScope/spectrascope_local/')
%load_ext autoreload
%autoreload 2

In [3]:
from sklearn.ensemble import RandomForestClassifier

In [4]:
from spectrascope.SpectraScope import *
from spectrascope.features import * 
from sklearn.linear_model import Ridge, Lasso, ElasticNet
import SPA 
import seaborn as sns
from spectrascope.ss_classification import *

In [5]:
from spectrascope import structure_property_featurization
from mp_api.client import MPRester
from spectrascope.structure_property_featurization import *
from collections import OrderedDict
from pymatgen.core.structure import Structure

In [10]:
#small dataset but doesn't do classification well 

fnX = "../tests/test_files/X_testfile.npy"
fny = "../tests/test_files/y_testfile.npy"
X = np.load(fnX)
y = np.load(fny)

X = X[:, :10, :]
y = y[:10]

structures = np.load("../tests/test_files/structures.npy", allow_pickle=True)
mpids = np.load("../tests/test_files/mpids.npy")
mpids = list(mpids)

info = BinaryClassifications(
    structures, mpids, X=X, y=y, specnames=["pdf", "xanes"], yname="nn", random_seed=10
)

saving to /examples/models, change save_path kwarg if youd like to change


0it [00:00, ?it/s]

In [7]:
# large dataset -- 12 minutes 

fnX = '/Users/amalyajohnson/Desktop/Spectrascope/spectrascope_local/tests/test_files/X_testfile.npy'
fny = '/Users/amalyajohnson/Desktop/Spectrascope/spectrascope_local/tests/test_files/y_testfile.npy'
X = np.load(fnX)
y = np.load(fny)

fn = '/Users/amalyajohnson/Desktop/Spectrascope/Datasets/XAS_PDF/Ti/processed_billinge.json'
data = pd.read_json(fn)
structures = data['structure'].apply(lambda x : Structure.from_dict(x))
structures = structures.to_numpy()
mpids = data['mp_id'].to_list()

info = BinaryClassifications(structures, mpids, X = X, y = y, specnames=['pdf', 'xanes'], yname='nn', random_seed=10, generate_dict=True)

In [11]:
info.plot_all(update_defaults=True)

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [12]:
info.structure_library_to_df()
info.structures_df.head()

,structure,is_triclinic,is_monoclinic,is_orthorhombic,is_tetragonal,is_trigonal,is_hexagonal,is_cubic,structure_vector,has_S,...,is_I4/mmm,is_R3c,is_Cmcm,is_Pmma,is_R3m,is_P1,is_R3,is_P2_1/c,is_P-1,"nn_in_[1.2,2]"
mp-1041683,"{'@module': 'pymatgen.core.structure', '@class...",True,False,False,False,False,False,False,"[0.12844639452727488, 0.1160463617286482, 0.12...",False,...,False,False,False,False,False,False,False,False,True,True
mp-556341,"{'@module': 'pymatgen.core.structure', '@class...",False,True,False,False,False,False,False,"[0.11619542458904952, 0.15710706439368743, 0.1...",True,...,False,False,False,False,False,False,False,True,False,False
mp-769525,"{'@module': 'pymatgen.core.structure', '@class...",False,False,False,False,True,False,False,"[0.11262424729476514, 0.10373667353938919, 0.1...",False,...,False,False,False,False,False,False,True,False,False,True
mp-677246,"{'@module': 'pymatgen.core.structure', '@class...",True,False,False,False,False,False,False,"[0.08122326446560829, 0.07795611310780487, 0.0...",False,...,False,False,False,False,False,True,False,False,False,True
mp-5020,"{'@module': 'pymatgen.core.structure', '@class...",False,False,False,False,True,False,False,"[0.09842201324211419, 0.08036123730020961, 0.0...",False,...,False,False,False,False,True,False,False,False,False,True


In [13]:
arr_entropy(info.ground_truth[0])

0.863120568566631

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
lr = LogisticRegression()
lr_pipeline_kwargs = {'estimator':lr, 
             'cv':2, 
             'param_grid':{'solver':['lbfgs'],
                           'penalty':['None']}, #,'l2', 'l1', 'elasticnet']},
             'refit':True,
             'metric':accuracy_score}

lr_kwargs = {'estimator':LogisticRegression(penalty=None, solver='lbfgs')}

In [15]:
info.add_x_axis(specname='xanes', value=np.linspace(4969.0, 5021.0, 200))
info.add_spectral_feature('xanes', 'pointwise_features', xrange=info.xanes_xaxis)

In [16]:
info.structure_library_to_df()

In [19]:
temp = StandardScaler().fit_transform(info.xanes_pointwise_features)
setattr(info, "xanes_pointwise_features_scaled", temp)

In [20]:
yattrs = [c for c in info.structures_df.columns if 'structure' not in c]
kwargs = dict(estimator=LogisticRegression(penalty=None, solver='lbfgs'), metric=accuracy_score)

# fails if using just 10 points because selected features is nothing
order =info.do_all_classification('xanes_pointwise_features_scaled', yattrs, savef=None, use_pipeline=False, scale_X=False, **kwargs)
# rf_predictions, order_rf =info.do_all_classification('xanes_pointwise_features', yattrs, savef=None, use_pipeline=False, estimator=RandomForestClassifier(), metric=accuracy_score)

IndexError: index 8 is out of bounds for axis 0 with size 1

In [22]:
from spectrascope.plot_spectral_features import PlotModel as psf 

In [18]:
fig = make_subplots(3,3, subplot_titles=order[1][:6], specs=[[{"secondary_y": True}, {"secondary_y": True}, {"secondary_y": True}], 
                                                          [{"secondary_y": True}, {"secondary_y": True}, {"secondary_y": True}], 
                                                          [{"secondary_y": True}, {"secondary_y": True}, {"secondary_y": True}],
                                                  ])
rows, cols = fig._get_subplot_rows_columns()
for ij, classification in zip(product(rows, cols), yattr_ordered[:9]): 
    row, col = ij 
    # print(classification)
    model = getattr(info, f"xanes_pointwise_features_scaled_{classification}_LogisticRegression")
    coefs = model['P'].coef_[0]
    best_feature_ids = np.argsort(coefs)[::-1][:20]
    coefs[coefs < 0] = 0 
    plotter = psf(fig=fig, raw=info.xanes_pointwise_features[0], selected_features=coefs, row=row, col=col)
    plotter.plot_pointwise_features()

fig.show()

NameError: name 'order' is not defined

In [48]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=info.xanes_xaxis, y=info.xanes_pointwise_features[0].ravel(), mode='lines'))
fig.update_yaxes(title_text='Intensity')
fig.update_xaxes(title_text='Energy (eV)')

In [385]:
train_ids, test_ids = train_test_split(np.arange(821), test_size=0.25, random_state=info.random_seed)
# true_mpids = [info.mpids[idx] for idx in test_ids]

in_top_10 = 0
in_top_5 = 0 
in_top_3 = 0 

id_set = test_ids
av = 0 
for idx in id_set: 
    classifications, top10 = info.find_similar_structures(idx, 'xanes_pointwise_features_scaled', yattrs, 'LogisticRegression', top=10)
    v2 = np.array([info.structures_df.loc[mpid]['structure_vector'] for mpid in top10])
    av += info.av_distance(v2)
        
    
    if info.mpids[idx] in top10: 
        in_top_10 +=1
        
print('% match in top10:', in_top_10 / len(id_set))
print('av similarity of top 10 structures', av/len(id_set))

% match in top20: 0.1262135922330097
av similarity of top 20 structures 0.48961728647085356


In [380]:
spec_classifications, top10 = info.find_similar_structures(35, 'xanes_pointwise_features_scaled', order[1], 'LogisticRegression')

In [381]:
## Distance of all original structures 
v = np.array([f for f in info.structures_df['structure_vector']])
info.av_distance(v)

0.8718025611634285

In [382]:
## Distance of top 10 similar structures 

v2 = np.array([info.structures_df.loc[mpid]['structure_vector'] for mpid in top10])
info.av_distance(v2)

0.6465993124615368

In [336]:
top_structures = [Structure.from_dict(info.structures_df.loc[mpid]['structure']) for mpid in top10]

In [383]:
top10

array(['mp-984604', 'mp-554704', 'mp-561121', 'mp-26630', 'mp-684119',
       'mp-603612', 'mp-25418', 'mp-684059', 'mp-560586', 'mp-557289'],
      dtype='<U10')

In [ ]:
['mp-603612', 'mp-6539', 'mp-6706', 'mp-554111', 'mp-6125',
       'mp-554704', 'mp-560586', 'mp-984604', 'mp-6268', 'mp-561121'],

In [342]:
# Not working 

import dash 
from dash import html, dcc
import crystal_toolkit

import crystal_toolkit.components as ctc


app = dash.Dash()

# structure_component1 = ctc.StructureMoleculeComponent(top_structures[0], id="structure 1")
# structure_component2 = ctc.StructureMoleculeComponent(top_structures[1], id="my_structure")
structure_components = [ctc.StructureMoleculeComponent(struct, id=f"structure_{i}") for i, struct in enumerate(top_structures)]

test = ctc.StructureMoleculeComponent(Structure.from_dict(top_structures[0]), id='hi')
# add the component's layout to our app's layout
# my_layout = html.Div([structure_component2.layout()])

my_layout = html.Div(className='row', children=[
    html.H1("Top 10 "),
    dcc.Dropdown(),
    html.Div(children=[
        test
    ])
])


ctc.register_crystal_toolkit(app=app, layout=my_layout)

if __name__ == "__main__":
    app.run_server(debug=True, port=8070)

TypeError: list indices must be integers or slices, not str

In [156]:
pd.DataFrame(np.array(spec_classifications).astype(bool), index=order).T

,is_C2/m,is_Cm,is_P-1,is_C2/c,is_Pnma,is_Cmcm,is_hexagonal,has_K,has_Si,is_orthorhombic,...,is_monoclinic,has_Li,is_trigonal,has_CN_5,has_P,has_oxidation_5.0,has_oxidation_3.0,has_oxidation_1.0,has_CN_2,has_CN_3
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,True,False,True,True,False


In [ ]:
feature_importances = getattr(info, 'xanes_pointwise_features_has_CN_4_RandomForestClassifier')['P'].feature_importances_
top = 100
sorted_best_features = np.argsort(feature_importances[::-1])[:top]

info.xanes_pointwise_features[:,sorted(sorted_best_features)].shape

In [ ]:
fig = make_subplots(1,2, subplot_titles=['ground_truth', 'prediction'])
fig.add_trace(go.Heatmap(z=result[:20,:20]), 1,1)
fig.add_trace(go.Heatmap(z=prediction[:20,:20]), 1,2)

In [ ]:
frames = [go.Frame(data=go.Heatmap(z=predictions[i])) for i in range(36)]

fig = go.Figure(frames=frames, data=go.Heatmap(z=predictions[0]))

animation_settings = {"frame":{"duration":50},
                      "mode":"immediate",
                      "transition":{"duration":50, "easing":"linear"}}

fig.update_layout(

    sliders=[{
        "steps": [{
            "args": [[frame.name], animation_settings],
            "label": str(k),
            "method": "animate"
        } for k, frame in enumerate(fig.frames)]
    }]
    
)

In [ ]:
fig.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
rfc = RandomForestClassifier()
rfc_kwargs = {'estimator':rfc,
              'cv':5, 
              'param_grid': {'n_estimators':[100, 200, 300], 
                             'max_depth':[35,50,None],
                             'criterion':['gini']},
              'refit':True,
            'metric':accuracy_score}

rf_kwargs = {'estimator':RandomForestRegressor(), 
                'param_grid':{'n_estimators':[50,100,200, 300], 
                                        'criterion':['squared_error'],
                                        'max_depth': [10,35, None]},
                'refit':True,
                'metric':mean_squared_error}

In [ ]:
from matminer.featurizers.structure import SiteStatsFingerprint 
from scipy.spatial.distance import cdist

cnnf = SiteStatsFingerprint.from_preset("CrystalNNFingerprint_ops")
df['structure'] = df['structure'].apply(lambda x: Structure.from_dict(x))
df['structure'] = df['structure'].apply(lambda x: x.add_oxidation_state_by_guess())